# Unit 2 solutions: instruments for the track

These reference solutions build each instrument, check it against a hand calculation or racekit's own number, and then point it at the starter.
Every race is deterministic or uses fixed seeds, every scratch file goes into a fresh temporary directory, and each answer is checked with executable asserts.
To keep the notebook fast, each race is run once and reused where the statements allow.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import KinematicCar
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import CONTROL_DT, run_race
from racekit.tracks import load_track

STARTER = Path("../../projects/racekit/agents/unit-01.py")
workdir = Path(mkdtemp(prefix="roar-unit-02-sol-"))
deterministic = lambda: KinematicCar(jitter=0.0)
print("scratch files go in", workdir)

## Exercise 1 - The car's view of three points

The car heads 170°, almost due west (−x).
A is about 19 m ahead and 6 m to the left, so the heading error is +18.5° and the starter steers left (a negative steer).
B is almost beside the car, 3.7 m ahead and 9.5 m to the right: −68.7°, a strong right steer.
C is 15 m **behind** the car and slightly right: −177.6°, which the clip turns into full right lock.
Unwrapped, the bearing to A (−171.5°) minus the yaw (170°) is −341.5°, which reads as "far to the right" and saturates the steer at full right lock, the wrong way for a point 18.5° to the left.

In [ ]:
def wrap_angle(angle):
    """Wrap an angle in radians to [-pi, pi)."""
    return (angle + math.pi) % (2.0 * math.pi) - math.pi


def to_car_frame(px, py, cx, cy, yaw):
    """The world point (px, py) seen from a car at (cx, cy) with heading yaw: (forward, left)."""
    dx, dy = px - cx, py - cy
    c, s = math.cos(yaw), math.sin(yaw)
    return c * dx + s * dy, -s * dx + c * dy


car_x, car_y, car_yaw = 120.0, -40.0, math.radians(170.0)
POINTS = {"A": (100.0, -43.0), "B": (118.0, -30.0), "C": (135.0, -42.0)}

car_frame = {name: to_car_frame(px, py, car_x, car_y, car_yaw) for name, (px, py) in POINTS.items()}
errors = {name: wrap_angle(math.atan2(py - car_y, px - car_x) - car_yaw) for name, (px, py) in POINTS.items()}
errors_deg = {name: math.degrees(error) for name, error in errors.items()}
naive_deg_A = math.degrees(math.atan2(POINTS["A"][1] - car_y, POINTS["A"][0] - car_x) - car_yaw)
behind = sorted(name for name, (forward, left) in car_frame.items() if forward < 0.0)
steer = {name: float(np.clip(-8.0 / math.sqrt(20.0) * error / math.pi, -1.0, 1.0)) for name, error in errors.items()}
which_way = (
    "The starter steers left toward A (error +18.5 deg, steer -0.18), right toward B (-68.7 deg, steer +0.68) "
    "and at full right lock toward C, which is behind the car (-177.6 deg); the naive error to A, -341.5 deg, "
    "is the same direction plus a whole extra turn, so without wrapping it reads as far to the right and "
    "saturates the steer at full right lock instead of a gentle left."
)

for name in POINTS:
    forward, left = car_frame[name]
    print(f"{name}: forward {forward:7.3f} m, left {left:7.3f} m | error {errors_deg[name]:8.3f} deg | steer {steer[name]:+.4f}")
print("naive error to A:", round(naive_deg_A, 3), "deg -> steer", float(np.clip(-8.0 / math.sqrt(20.0) * math.radians(naive_deg_A) / math.pi, -1.0, 1.0)))
print("behind:", behind)

# The heading error in the car frame is atan2(left, forward); it must agree with the world-frame one.
for name, (forward, left) in car_frame.items():
    assert abs(wrap_angle(math.atan2(left, forward) - errors[name])) < 1e-12
assert -180.0 <= min(errors_deg.values()) and max(errors_deg.values()) < 180.0
assert behind == ["C"] and car_frame["A"][1] > 0.0 > car_frame["B"][1]
assert steer["A"] < 0.0 < steer["B"] < 1.0 and steer["C"] == 1.0
assert abs(wrap_angle(math.radians(naive_deg_A)) - errors["A"]) < 1e-12 and naive_deg_A < -180.0
assert float(np.clip(-8.0 / math.sqrt(20.0) * math.radians(naive_deg_A) / math.pi, -1.0, 1.0)) == 1.0  # full right lock

## Exercise 2 - Differencing the yaw column

The raw column stays in [−π, π), so whenever the heading crosses ±π the difference of two neighbouring rows jumps by almost 2π: one tick of that is a "rate" of 2π / 0.05 ≈ 126 rad/s.
Wrapped, the largest rate is about 0.17 rad/s, and the median rate in the bends matches v / R = 19.49 / 120 ≈ 0.1624 rad/s to four decimals.
Summed, the wrapped steps add up to almost exactly three turns of 2π (three laps counter-clockwise; the last row stops just short of a whole third turn).

In [ ]:
oval_race = run_race(STARTER, "oval", car=deterministic())
oval_yaw = oval_race.telemetry["yaw"]
raw_steps = np.diff(oval_yaw)
naive_max_rate = float(np.max(abs(raw_steps / CONTROL_DT)))
wrapped_steps = np.array([wrap_angle(float(step)) for step in raw_steps])
wrapped_rates = wrapped_steps / CONTROL_DT
wrapped_max_rate = float(np.max(abs(wrapped_rates)))
wrap_jumps = int(np.sum(abs(raw_steps) > math.pi))
total_turn = float(np.sum(wrapped_steps))
laps_turned = total_turn / (2.0 * math.pi)
bend_rate = float(np.median(abs(wrapped_rates)[abs(wrapped_rates) > 0.1]))
bend_rate_prediction = 19.49 / 120.0
why_jumps = (
    "The yaw column is stored wrapped to [-pi, pi), and on the oval's westbound straight the car heads "
    "almost exactly at +-pi, so every tiny wobble across that heading flips the stored value by nearly 2 pi "
    "between two rows, although the car itself only turned a fraction of a degree."
)

print(oval_race)
print(f"naive max rate {naive_max_rate:.3f} rad/s | wrapped max rate {wrapped_max_rate:.5f} rad/s | jumps {wrap_jumps}")
print(f"total turn {total_turn:.5f} rad = {laps_turned:.5f} turns | bend rate {bend_rate:.5f} rad/s vs v/R {bend_rate_prediction:.5f} rad/s")
assert abs(oval_race.total_time - 223.50) < 1e-9 and oval_race.collisions == 0
assert len(wrapped_rates) == len(oval_yaw) - 1 and np.max(abs(wrapped_steps)) <= math.pi
assert abs(naive_max_rate - 2.0 * math.pi / CONTROL_DT) < 0.01  # one near-2 pi flip in a single tick
assert np.min(abs(oval_yaw[1:][abs(raw_steps) > math.pi])) > 3.1  # every jump happens at a heading near +-pi
assert wrapped_max_rate < 0.2 and abs(laps_turned - 3.0) < 0.01
assert abs(bend_rate - bend_rate_prediction) < 0.001

## Exercise 3 - Projection, and a stale hint

By hand, for a = (2, 1) → b = (10, 1): d = (8, 0) and |d|² = 64.
For p = (6, 5), e = (4, 4), so t = 32 / 64 = 0.5 (inside the segment), the foot is (6, 1) and the distance is 4.
For q = (−1, −3), e = (−3, −4), so t = −24 / 64 = −0.375; clamped to 0 the foot is a itself, and the distance is |q − a| = 5.

From hint 600 the window contains the right segment, and the search agrees with brute force: segment 600, 1 m away (the point was built 1 m to its left).
From hint 0 or 585 the window does not contain segment 600, and the search returns the nearest segment it *can* see (8 and 593, the last segment of each window) without any error.
The giveaway is the distance to the foot: hundreds of metres, or 12.6 m, against the 1 m a correct answer gives, and more than a half-width.

In [ ]:
def project_onto_segment(px, py, ax, ay, bx, by):
    """(t clamped to [0, 1], squared distance from p to the clamped foot) for the segment a -> b."""
    dx, dy = bx - ax, by - ay
    t = ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)
    t = max(0.0, min(t, 1.0))
    fx, fy = px - (ax + t * dx), py - (ay + t * dy)
    return t, fx * fx + fy * fy


def nearest_segment(px, py, xs, ys, hint, reach=8):
    """The windowed search: the segment in hint - reach .. hint + reach (cyclic) whose clamped foot is nearest."""
    n = len(xs)
    best_d2, best = math.inf, hint % n
    for k in range(hint - reach, hint + reach + 1):
        i = k % n
        j = (i + 1) % n
        _, d2 = project_onto_segment(px, py, xs[i], ys[i], xs[j], ys[j])
        if d2 < best_d2:
            best_d2, best = d2, i
    return best


def raw_t(px, py, ax, ay, bx, by):
    dx, dy = bx - ax, by - ay
    return ((px - ax) * dx + (py - ay) * dy) / (dx * dx + dy * dy)


# (a) By hand: d = (8, 0), |d|^2 = 64.
# p = (6, 5): e = (4, 4), t = 32 / 64 = 0.5, the foot is (6, 1), the distance is 4.
# q = (-1, -3): e = (-3, -4), t = -24 / 64 = -0.375, clamped to 0, so the foot is a = (2, 1), the distance is 5.
t_raw_p = raw_t(6.0, 5.0, 2.0, 1.0, 10.0, 1.0)
t_p, d2_p = project_onto_segment(6.0, 5.0, 2.0, 1.0, 10.0, 1.0)
dist_p = math.sqrt(d2_p)
t_raw_q = raw_t(-1.0, -3.0, 2.0, 1.0, 10.0, 1.0)
t_q, d2_q = project_onto_segment(-1.0, -3.0, 2.0, 1.0, 10.0, 1.0)
dist_q = math.sqrt(d2_q)
assert (t_raw_p, t_p, dist_p) == (0.5, 0.5, 4.0)
assert (t_raw_q, t_q, dist_q) == (-0.375, 0.0, 5.0)

tc = load_track("test-circuit")
tc_x, tc_y = [float(value) for value in tc.x], [float(value) for value in tc.y]
dx, dy = tc_x[601] - tc_x[600], tc_y[601] - tc_y[600]
length = math.hypot(dx, dy)
px = tc_x[600] + 0.3 * dx - dy / length
py = tc_y[600] + 0.3 * dy + dx / length

n_tc = len(tc_x)
found = {}
foot_distance = {}
for hint in [600, 0, 585]:
    found[hint] = nearest_segment(px, py, tc_x, tc_y, hint)
    j = (found[hint] + 1) % n_tc
    foot_distance[hint] = math.sqrt(project_onto_segment(px, py, tc_x[found[hint]], tc_y[found[hint]], tc_x[j], tc_y[j])[1])
brute = min(range(n_tc), key=lambda k: project_onto_segment(px, py, tc_x[k], tc_y[k], tc_x[(k + 1) % n_tc], tc_y[(k + 1) % n_tc])[1])
silent_failure = (
    "When the window does not contain the right segment the search still returns a segment, the nearest one "
    "inside the window (here the window's last segment), with no warning; foot_distance exposes it, because "
    "the true answer is 1 m away while the stale answers are 12.6 m and 634 m away, more than the track's half-width."
)

print("found:", found)
print("foot distance:", {hint: round(value, 4) for hint, value in foot_distance.items()})
print("brute force:", brute)
assert found[600] == brute == 600 and abs(foot_distance[600] - 1.0) < 1e-9
assert found[0] == 8 and found[585] == 593  # the edge of each stale window
assert foot_distance[0] > foot_distance[585] > float(np.max(tc.half_width))

## Exercise 4 - The cross-track error of the starter on `test-circuit`

The instrument reproduces racekit's `offset` on every row to float rounding.
The starter stays within about half a metre of the centreline, and it is more often slightly right than left.
`test-circuit` runs counter-clockwise, so most corners are left-handers, and running wide in a left-hander means sitting to the right: that is why the biggest excursion (−0.51 m, in the R = 18 m hairpin) is to the right.
The last check confirms the lesson's finding with the generator's curvature: in every corner row the offset has the opposite sign of κ, the outside of the corner.

In [ ]:
def signed_offset(px, py, ax, ay, bx, by):
    """Signed distance from p to the line through a -> b, left positive."""
    dx, dy = bx - ax, by - ay
    ex, ey = px - ax, py - ay
    return (dx * ey - dy * ex) / math.sqrt(dx * dx + dy * dy)


def cross_track(px, py, xs, ys, hint, reach=8):
    """(segment index, signed cross-track error): the windowed search, then the offset to the chosen segment."""
    i = nearest_segment(px, py, xs, ys, hint, reach)
    j = (i + 1) % len(xs)
    return i, signed_offset(px, py, xs[i], ys[i], xs[j], ys[j])


def track_position(telemetry, xs, ys):
    """cross_track over every row, the hint carried from 0 (a held row restarts it at 0): arrays (segment, offset)."""
    hint, segments, offsets = 0, [], []
    for x, y, held in zip(telemetry["x"], telemetry["y"], telemetry["held"]):
        hint, offset = cross_track(float(x), float(y), xs, ys, 0 if held else hint)
        segments += [hint]
        offsets += [offset]
    return np.array(segments), np.array(offsets)


tc_race = run_race(STARTER, "test-circuit", car=deterministic())
my_segments, my_offsets = track_position(tc_race.telemetry, tc_x, tc_y)
max_disagreement = float(np.max(abs(my_offsets - tc_race.telemetry["offset"])))
max_left = float(np.max(my_offsets))
max_right = float(np.min(my_offsets))
left_fraction = float(np.mean(my_offsets > 0.0))
min_margin = float(np.min(tc.half_width[my_segments] - abs(my_offsets)))
tc_corner = abs(tc.curvature[my_segments]) > 0.01
outside_of_corners = bool(np.max(my_offsets[tc_corner] * tc.curvature[my_segments][tc_corner]) < 0.0)

print(tc_race, "|", len(my_offsets), "rows")
print(f"max disagreement {max_disagreement:.2e} m | max left {max_left:.4f} m | max right {max_right:.4f} m")
print(f"left fraction {left_fraction:.4f} | min margin {min_margin:.4f} m | outside of corners: {outside_of_corners}")
assert tc_race.finished and tc_race.collisions == 0 and abs(tc_race.total_time - 412.75) < 1e-9
assert max_disagreement < 1e-6
assert len(my_offsets) == len(tc_race.telemetry) and np.max(tc_race.telemetry["held"]) == 0
assert abs(max(max_left, -max_right) - 0.51) < 0.005  # the lesson's peak, in the hairpin
assert 0.0 < left_fraction < 0.5 and min_margin > 4.9
assert outside_of_corners

## Exercise 5 - Curvature profiles, and a centimetre of noise

By hand: twice the signed area is (2, 0.01) × (4, 0) = 2 · 0 − 0.01 · 4 = −0.04, and the sides are about 2, 2 and 4, so κ ≈ 2 · (−0.04) / 16 = −0.005 1/m.
Moving the middle point to the left makes the path bend **right** at the ends, so the sign is negative, and the invented "corner" has a radius of about 200 m: as tight as many real corners.

The profiles reproduce the lesson's radii (120, 18 and 25 m) and the ±2π total turning.
The `oval` has no corner tighter than 100 m at all: its bends are R = 120 m, κ ≈ 0.0083 < 0.01.
Runs of |κ| > 0.01 are counted where a tight waypoint follows a non-tight one; Python's `tight[-1]` is the last waypoint, so a run across the start line is counted once.

In [ ]:
def menger_curvature(ax, ay, bx, by, cx, cy):
    """Signed curvature of the circle through a, b, c: 4 * area / (product of the sides), left turns positive."""
    cross = (bx - ax) * (cy - ay) - (by - ay) * (cx - ax)  # twice the signed area
    sides = math.hypot(bx - ax, by - ay) * math.hypot(cx - bx, cy - by) * math.hypot(ax - cx, ay - cy)
    return 2.0 * cross / sides


def curvature_profile(xs, ys):
    """The signed Menger curvature at every waypoint of a closed track."""
    n = len(xs)
    return np.array([menger_curvature(xs[i - 1], ys[i - 1], xs[i], ys[i], xs[(i + 1) % n], ys[(i + 1) % n]) for i in range(n)])


# (a) By hand: twice the area is (2, 0.01) x (4, 0) = -0.04; the sides are about 2, 2 and 4, so kappa ~ 2 * -0.04 / 16 = -0.005.
kappa_bump = menger_curvature(-2.0, 0.0, 0.0, 0.01, 2.0, 0.0)
radius_bump = 1.0 / abs(kappa_bump)

r_min, straight_fraction, total_turning, direction, corners = {}, {}, {}, {}, {}
kappas = {}
for name in ["oval", "test-circuit", "monza-like"]:
    track = load_track(name)
    xs, ys = [float(value) for value in track.x], [float(value) for value in track.y]
    kappa = curvature_profile(xs, ys)
    kappas[name] = kappa
    n = len(xs)
    ds = np.array([math.hypot(xs[(i + 1) % n] - xs[i], ys[(i + 1) % n] - ys[i]) for i in range(n)])
    r_min[name] = float(1.0 / np.max(abs(kappa)))
    straight_fraction[name] = float(np.mean(abs(kappa) < 0.001))
    total_turning[name] = float(np.sum(kappa * ds))
    direction[name] = "counter-clockwise" if total_turning[name] > 0.0 else "clockwise"
    tight = abs(kappa) > 0.01
    corners[name] = int(sum(1 for i in range(n) if tight[i] and not tight[i - 1]))  # tight[-1] closes the loop
noise_lesson = (
    "A one-centimetre error over 2 m spacing already invents a 200 m corner, so curvature computed point by "
    "point from measured positions (such as a car's recorded path) is dominated by noise; it must be smoothed "
    "or measured over points much further apart than 2 m before it means anything."
)

print(f"kappa_bump {kappa_bump:.6f} 1/m | radius_bump {radius_bump:.3f} m")
for name in r_min:
    print(f"{name:>12}: R_min {r_min[name]:7.3f} m | straight {straight_fraction[name]:.4f} | turning {total_turning[name]:+.5f} rad "
          f"({direction[name]}) | corners {corners[name]}")
assert kappa_bump < 0.0 and abs(radius_bump - 200.0) < 0.01
assert abs(r_min["oval"] - 120.0) < 0.005 and abs(r_min["test-circuit"] - 18.0) < 0.005 and abs(r_min["monza-like"] - 25.0) < 0.005
assert max(abs(abs(turning) - 2.0 * math.pi) for turning in total_turning.values()) < 0.01
assert direction == {"oval": "counter-clockwise", "test-circuit": "counter-clockwise", "monza-like": "clockwise"}
assert corners["oval"] == 0 and corners["test-circuit"] > 0 and corners["monza-like"] > 0
assert np.min(abs(kappas["monza-like"])) <= 0.01 and np.min(abs(kappas["test-circuit"])) <= 0.01  # no run wraps the whole lap

## Exercise 6 - Does a stiffer gain follow the line better, and is it faster?

Halving the gain doubles the RMS offset (about 0.14 m to 0.30 m), a gap far wider than either set's range, so gain 8 does follow the line better.
The times differ by only 0.25 s in the median, less than the 0.35 s spread, so the rule cannot call gain 8 faster.
On the oval at 20 m/s, a small offset is cheap: a few tenths of a metre off the line barely changes the distance driven.

In [ ]:
variant_4 = workdir / "gain_4.py"
write_variant(STARTER, variant_4, {"STEER_GAIN": "4.0"})
jittered = lambda: KinematicCar(jitter=0.05)
runs_8 = repeat_races(STARTER, "oval", n=5, car_factory=jittered)
runs_4 = repeat_races(variant_4, "oval", n=5, car_factory=jittered)


def offset_stats(race):
    driving = abs(race.telemetry["offset"][race.telemetry["held"] == 0])
    return float(np.sqrt(np.mean(driving ** 2))), float(np.max(driving))


rms_8, rms_4 = [], []
max_8, max_4 = [], []
for race in runs_8.results:
    rms, peak = offset_stats(race)
    rms_8 += [rms]
    max_8 += [peak]
for race in runs_4.results:
    rms, peak = offset_stats(race)
    rms_4 += [rms]
    max_4 += [peak]
rms_median_8, rms_range_8 = float(np.median(rms_8)), max(rms_8) - min(rms_8)
rms_median_4, rms_range_4 = float(np.median(rms_4)), max(rms_4) - min(rms_4)
follows_better = noise_aware_better(rms_8, rms_4)
faster = noise_aware_better(runs_8.times, runs_4.times)

fig, ax = plt.subplots(figsize=(11, 4))
for gain, runs in [(8, runs_8), (4, runs_4)]:
    tel = runs.results[0].telemetry
    ax.plot(tel["t"], tel["offset"], linewidth=0.8, label=f"STEER_GAIN {gain}")
t_all = runs_8.results[0].telemetry["t"]
ax.fill_between(t_all, -rms_median_8, rms_median_8, color="tab:green", alpha=0.2, label=f"± gain-8 RMS median ({rms_median_8:.3f} m)")
ax.set_title("Starter on the oval, seed 0: cross-track offset for two steering gains")
ax.set_xlabel("t (s)")
ax.set_ylabel("offset (m, left +)")
ax.legend()
fig.tight_layout()
plt.show()
verdict = (
    "Gain 8 follows the line clearly better (half the RMS offset, far beyond the noise), but it is not "
    "provably faster, because a few tenths of a metre off the centreline cost almost nothing at 20 m/s on "
    "the oval: following the line better is not the same as going faster."
)

print("gain 8:", runs_8, np.round(runs_8.times, 2))
print("gain 4:", runs_4, np.round(runs_4.times, 2))
print("rms 8:", np.round(rms_8, 4), "| rms 4:", np.round(rms_4, 4))
print("max 8:", np.round(max_8, 4), "| max 4:", np.round(max_4, 4))
print(f"rms median/range 8: {rms_median_8:.4f}/{rms_range_8:.4f} | 4: {rms_median_4:.4f}/{rms_range_4:.4f}")
print("follows better:", follows_better, "| faster:", faster)
assert runs_8.all_finished and runs_4.all_finished and runs_8.collisions == 0 and runs_4.collisions == 0
assert abs(runs_8.median - 220.75) < 1e-9  # Unit 1's seeds 0-4
assert follows_better and not noise_aware_better(rms_4, rms_8)
assert not faster and not noise_aware_better(runs_4.times, runs_8.times)
assert min(peak_4 - peak_8 for peak_4, peak_8 in zip(max_4, max_8)) > 0.0

## Exercise 7 - Challenge: the sign of a corner, and the sign of the steer

About two thirds of `monza-like`'s corner rows are right-handers (κ < 0), as a clockwise track should have.
In **every** corner row the steer and κ have opposite signs: a left turn (κ > 0) needs a left steer, which is negative in ROAR's convention.
The ratio is steady, about −2.37 m: the steer is roughly proportional to the curvature, so a corner of radius R needs a steer of about −2.37 / R (−0.095 for R = 25 m).

In [ ]:
monza = load_track("monza-like")
mx, my = [float(value) for value in monza.x], [float(value) for value in monza.y]
monza_kappa = kappas["monza-like"]
monza_race = run_race(STARTER, "monza-like", car=deterministic())
monza_segments, monza_offsets = track_position(monza_race.telemetry, mx, my)
row_kappa = monza_kappa[monza_segments]
corner = abs(row_kappa) > 0.01
corner_steer = monza_race.telemetry["steer"][corner]
corner_kappa = row_kappa[corner]
corner_rows = int(np.sum(corner))
right_turn_fraction = float(np.mean(corner_kappa < 0.0))
opposite_sign_fraction = float(np.mean(corner_steer * corner_kappa < 0.0))
steer_per_kappa = float(np.median(corner_steer / corner_kappa))
convention = (
    "A left turn has positive curvature but needs a left steer, which is negative in ROAR's convention, so a "
    "correct controller always steers with the opposite sign of the corner; the steady ratio of about -2.37 m "
    "means a corner of radius R needs a steer of about -2.37 / R, larger for tighter corners."
)

print(monza_race)
print(f"corner rows {corner_rows} | right-turn fraction {right_turn_fraction:.4f} | opposite sign {opposite_sign_fraction:.4f} | steer/kappa {steer_per_kappa:.4f} m")
assert monza_race.finished and monza_race.collisions == 0 and abs(monza_race.total_time - 882.55) < 1e-9
assert np.max(abs(monza_offsets - monza_race.telemetry["offset"])) < 1e-6
assert right_turn_fraction > 0.5  # a clockwise track turns mostly right
assert opposite_sign_fraction == 1.0 and steer_per_kappa < 0.0

## Exercise 8 - Challenge: curvature and offset at 30 m/s

At both speeds |offset| follows |κ| closely (r ≈ 0.975 under the car), and the correlation peaks at shift −2: with the curvature two waypoints, about 4 m, **behind** the car.
The offset lags the corner: the car runs widest a little after it reaches each piece of curve, because the error builds up while the corner pulls the track away.
At 30 m/s the race is still clean, the shape of the correlation curve is almost identical (same best shift, r ≈ 0.995), and only the size grows: the peak offset is about 1.48 times larger.

In [ ]:
def pearson(a, b):
    da, db = a - np.mean(a), b - np.mean(b)
    return float(np.sum(da * db) / np.sqrt(np.sum(da ** 2) * np.sum(db ** 2)))


race_20 = monza_race  # the same deterministic 20 m/s race as Exercise 7
race_30 = run_race(write_variant(STARTER, workdir / "speed_30.py", {"TARGET_SPEED": "30.0"}), "monza-like", car=deterministic())
clean_30 = race_30.finished and race_30.collisions == 0
n_monza = len(mx)
r_at_car, best_shift, r_best, max_offset = {}, {}, {}, {}
correlations = {}
for speed, race in [(20, race_20), (30, race_30)]:
    segments, offsets = (monza_segments, monza_offsets) if speed == 20 else track_position(race.telemetry, mx, my)
    correlations[speed] = {shift: pearson(abs(offsets), abs(monza_kappa[(segments + shift) % n_monza])) for shift in range(-10, 11)}
    r_at_car[speed] = correlations[speed][0]
    best_shift[speed] = max(correlations[speed], key=lambda shift: correlations[speed][shift])
    r_best[speed] = correlations[speed][best_shift[speed]]
    max_offset[speed] = float(np.max(abs(offsets)))
offset_ratio = max_offset[30] / max_offset[20]
reading = (
    "The best shift is negative (-2 at both speeds): the offset matches the curvature about 4 m behind the "
    "car, so the car runs widest just after it has entered each piece of curve, lagging the corner. "
    "At 30 m/s the correlation curve and its best shift are almost unchanged, so 50% more speed changes only "
    "the size of the excursion (about 1.5 times larger), not the relation between curvature and offset."
)

print(race_20, "|", race_30, "| clean 30:", clean_30)
for speed in [20, 30]:
    print(speed, {shift: round(r, 4) for shift, r in correlations[speed].items()})
print("r at car:", r_at_car, "| best shift:", best_shift, "| r best:", r_best)
print("max offset:", max_offset, "| ratio:", offset_ratio)
assert clean_30 and race_30.total_time < race_20.total_time
assert np.max(race_30.telemetry["held"]) == 0  # no respawn hold: every row is driving
assert best_shift[20] == best_shift[30] < 0
assert min(r_best[speed] - r_at_car[speed] for speed in [20, 30]) >= 0.0 and min(r_at_car.values()) > 0.9
assert 1.0 < offset_ratio < 2.25